# Lecture 9 - The agentic workflow

**Lecture Topic**: Tool calling and the anatomy of a harness, then going from a vague request to a precise spec

## Announcements

- Project proposals (Milestone 1) are due this Friday, Oct 16 at 11:59pm. If you still don't have a group, please let me know today.
- There is no class on Friday.
- ClassBuzz poll for today: `XXXX`

## Recap

In the last lecture, we covered the following topics:
 - Broadcasting by name and groupby in xarray
 - An LLM predicts the next token, and tokens aren't words
 - Sampling and temperature
 - Training, knowledge cutoffs, and open versus closed models
 - Chat templates, the cost of re-sending the conversation, and context windows

Today, we will cover the following topics:
 - Tool calling and the agent loop
 - The anatomy of a harness, and a real Claude Code session traced step by step
 - The agentic workflow: requirements, spec, plan, implement, verify
 - The same task from a vague request and from a precise spec
 - Writing a spec

Let's load our small open-weights model again, along with the `ask` helper from last time:

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import mlx.core as mx
from mlx_lm import load, generate
from mlx_lm.sample_utils import make_sampler

model, tokenizer = load("mlx-community/Qwen3-4B-Instruct-2507-4bit")

def ask(question, temperature=0.0, max_tokens=100):
    messages = [{"role": "user", "content": question}]
    prompt = tokenizer.apply_chat_template(messages, add_generation_prompt=True)
    return generate(model, tokenizer, prompt=prompt, max_tokens=max_tokens,
                    sampler=make_sampler(temp=temperature))

## Tool calling

So far the model can only produce text. How does an agent run code or edit files?

The harness describes the available **tools** in the prompt, and the model has been trained to write a special kind of text when it wants to use one. Here's a tool to multiply two numbers (the docstring and type hints become the description):

In [ ]:
def multiply(a: float, b: float):
    """
    A function that multiplies two numbers

    Args:
        a: The first number to multiply
        b: The second number to multiply
    """
    return a * b

messages = [{"role": "user", "content": "What is 12234585 times 48838483920?"}]
prompt = tokenizer.apply_chat_template(messages, tools=[multiply], add_generation_prompt=True)
print(tokenizer.decode(prompt))

The tool has been turned into a JSON description in the system prompt. Let's see what the model writes back:

In [ ]:
reply = generate(model, tokenizer, prompt=prompt, max_tokens=100)
print(reply)

The model **doesn't run anything**. It just writes text asking for a tool call. It's up to the harness to spot that, run the function, and paste the result back into the conversation as a new message.

(Which is just as well, since without the tool it gets the answer wrong:)

In [ ]:
print(ask("What is 12234585 times 48838483920? Just the number."))
print(12234585 * 48838483920)

## The agent loop

Putting that together gives us an **agent**: keep calling the model, running any tools it asks for, until it answers without asking for one.

Let's give our model a tool that runs Python, with the taxi data already loaded:

In [ ]:
import io, contextlib, json, re

taxis = pd.read_csv('../data/2019_Yellow_Taxi_Trip_Data.csv')

def run_python(code: str):
    """
    Run Python code and return whatever it prints. A pandas DataFrame called
    `taxis` (NYC taxi trips) is already loaded, and pandas is imported as pd.

    Args:
        code: The Python code to run
    """
    out = io.StringIO()
    try:
        with contextlib.redirect_stdout(out):
            exec(code, {"taxis": taxis, "pd": pd})
    except Exception as e:
        return f"Error: {e!r}"
    return out.getvalue()

In [ ]:
def agent(question, tools, system=None, max_steps=6):
    messages = [{"role": "system", "content": system}] if system else []
    messages.append({"role": "user", "content": question})
    for step in range(max_steps):
        prompt = tokenizer.apply_chat_template(messages, tools=list(tools.values()), add_generation_prompt=True)
        reply = generate(model, tokenizer, prompt=prompt, max_tokens=500)
        messages.append({"role": "assistant", "content": reply})
        calls = re.findall(r"<tool_call>(.*?)</tool_call>", reply, re.S)
        if not calls:
            return reply  # no tool requested, so we're done
        for call in calls:
            call = json.loads(call)
            print(f"[step {step}] model asks to run:\n{call['arguments']['code']}")
            result = tools[call["name"]](**call["arguments"])  # the harness runs it
            print(f"[step {step}] harness sends back: {result!r}\n")
            messages.append({"role": "tool", "content": result})

That's about 15 lines. Real harnesses are much bigger, but this is the core of every coding agent.

### Quick quiz

What will our agent answer to *"Which payment type has the highest average tip?"*

Recall that `payment_type` is 1 for card and 2 for cash, and cash tips are always recorded as 0.

 - A. Card
 - B. Cash
 - C. It will crash
 - D. It will say it can't tell

In [ ]:
agent("Which payment type has the highest average tip in the taxi data?", tools={"run_python": run_python})

The code was fine, but it didn't `print` anything, so the tool sent back an empty string. The model then made up an answer that sounds plausible (and is wrong in a way you wouldn't spot without looking at the data).

### Improving the harness

Let's not touch the model at all. Instead we'll change the tool to behave like a Jupyter cell (show the value of the last line, and say so if there's no output), and add a system prompt with a short data dictionary:

In [ ]:
import ast

def run_python(code: str):
    """
    Run Python code and return the output, like a Jupyter cell. A pandas
    DataFrame called `taxis` (NYC taxi trips) is already loaded, and pandas
    is imported as pd.

    Args:
        code: The Python code to run
    """
    out = io.StringIO()
    env = {"taxis": taxis, "pd": pd}
    try:
        tree = ast.parse(code)
        last = tree.body.pop() if tree.body and isinstance(tree.body[-1], ast.Expr) else None
        with contextlib.redirect_stdout(out):
            exec(compile(tree, "<agent>", "exec"), env)
            if last is not None:  # print the value of the last line, like Jupyter does
                print(repr(eval(compile(ast.Expression(last.value), "<agent>", "eval"), env)))
    except Exception as e:
        return f"Error: {e!r}"
    return out.getvalue() or "(the code ran but produced no output)"

SYSTEM = """You are a data analysis assistant. Answer questions using only the `taxis` DataFrame,
via the run_python tool. Never make up or simulate data.
payment_type: 1 = card, 2 = cash (cash tips are not recorded), 3 = no charge, 4 = dispute."""

In [ ]:
agent("Which payment type has the highest average tip in the taxi data?",
      tools={"run_python": run_python}, system=SYSTEM)

Same model, same question, correct answer. The only things we changed were what the tool sends back and the system prompt.

A 4B model is still a long way behind the frontier models (ask it something harder and it soon falls apart), but the harness clearly matters as much as the model!

## The anatomy of a harness

A production harness like Claude Code, Codex or Copilot has the same loop at its core, plus a lot of engineering around it:

| Part | What it does | In Claude Code |
|---|---|---|
| **System prompt** | How to behave, the date, the OS, the git status... | ~2,300 tokens |
| **Tools** | Read, edit and write files; run shell commands; search; fetch web pages | `Read`, `Edit`, `Write`, `Bash`, `Grep`, `Glob`, `WebFetch`, `WebSearch`, ... |
| **Project instructions** | Notes about *your* project, loaded into every session | `CLAUDE.md` (most other agents use `AGENTS.md`) |
| **Permissions** | Which tools can run without asking you | Modes from "ask for everything" to "auto" (a second model reviews each action) |
| **Context management** | Keeping the context small and relevant | Sub-agents with their own context, and summarising ("compacting") old history |
| **Extensions** | Connecting other tools and services | MCP servers, skills, hooks |

### Same model, different harness

You can see this in the benchmarks. On Terminal-Bench 2.1 (a set of realistic command-line tasks), the same model gets quite different scores depending on the harness it runs in. For example, GPT-5.5 scored 76% in one harness and 57% in another ([Vals.ai, Sep 2026](https://www.vals.ai/benchmarks/terminal-bench-2-1)).

So when you see "model X scores Y%", it's worth asking: in which harness, and with what settings?

### Open-source harnesses

Several good harnesses are open source, and most of them can use an open-weights model running on your own machine (e.g. through Ollama) instead of a paid API:

 - [OpenAI Codex CLI](https://github.com/openai/codex) (Apache 2.0)
 - [OpenCode](https://opencode.ai) (MIT)
 - [Aider](https://aider.chat), [Goose](https://github.com/aaif-goose/goose), [Cline](https://github.com/cline/cline), [Qwen Code](https://github.com/QwenLM/qwen-code), [Gemini CLI](https://github.com/google-gemini/gemini-cli)

*Tip: verified students get GitHub Copilot's student plan for free through [GitHub Education](https://github.com/settings/education/benefits), which may be useful for your project.*

## A real session, step by step

Let's go back to the tip analysis from Lecture 1. I set up a small project with the agent's original code in `tips.py`:

```python
def top_tip_zones(df, n=5):
    """Return the n pickup zones with the highest mean tip percentage."""
    df = df.copy()
    df["tip_pct"] = df["tip_amount"] / df["total_amount"] * 100
    return (df.groupby("pulocationid")["tip_pct"]
              .mean().sort_values(ascending=False).head(n))
```

and a `CLAUDE.md`:

```markdown
# Taxi tips analysis
- Python lives at ~/miniforge3/envs/dsc200-fa26/bin/python (pandas 3). pytest is not installed: write tests
  as plain `assert`s in `test_*.py` files and run them with that Python.
- Data: data/taxi_2019.csv, a 10,000-row sample of 2019 NYC yellow-taxi trips.
  payment_type: 1 = card, 2 = cash, 3 = no charge, 4 = dispute.
```

Then I asked Claude Code (Opus 5.5), allowing it to edit files and run Python, but nothing else without asking:

> *"The tip percentages from tips.py look off to me - I suspect cash trips are part of the problem. Can you fix top_tip_zones so the answer is meaningful, and add a test?"*

Claude Code can save every session as a log file, one JSON object per line. Let's read it in:

In [ ]:
session = [json.loads(line) for line in open('../data/claude_code_session.jsonl')]
init = session[0]
print(init['model'], init['permissionMode'])
print(len(init['tools']), 'tools:', ', '.join(init['tools']))

### What's in the context before I've typed anything?

Claude Code's `/context` command shows what the model will read before my prompt (this is for a fresh session in that folder):

| Category | Tokens |
|---|---|
| System prompt | 2.3k |
| Tool descriptions | 8.3k |
| Skills | 2.1k |
| Memory files (`CLAUDE.md`) | 0.2k |
| Tools that are only loaded when needed | (13.3k, not loaded) |
| **Total** | **12.9k of 1M** |

So roughly 13,000 tokens go in before it has done anything, and they get sent again on every step.

### The steps

Let's print each tool call and a short version of what came back:

In [ ]:
def short(text, n=300):
    text = text if isinstance(text, str) else json.dumps(text)
    return text if len(text) <= n else text[:n] + ' ...'

for event in session:
    if event['type'] == 'assistant':
        for block in event['message']['content']:
            if block['type'] == 'tool_use':
                args = block['input'].get('command') or block['input'].get('file_path')
                print(f"MODEL -> {block['name']}: {short(args, 200)}")
    elif event['type'] == 'user':
        for block in event['message']['content']:
            if block['type'] == 'tool_result':
                print(f"   HARNESS <- {'ERROR ' if block.get('is_error') else ''}{short(block['content'], 120)}\n")
    elif event['type'] == 'system' and event.get('subtype') == 'permission_denied':
        print(f"   PERMISSION DENIED: {event['message']}")

### What it did

1. **Looked around** first: listed the files, read `tips.py` and the first lines of the data. (The command "failed" only because there were no test files yet, and it carried on.)
1. **Explored the data** with two short Python scripts: tips by payment type, how many trips are behind the top zones, and the odd rows with negative totals.
1. **Wrote** the new `tips.py` and `test_tips.py`.
1. Tried to check that the tests would *fail* on the old code using `git stash`. I hadn't allowed git commands that change files, so **the harness blocked it**, and the model ran just the tests instead.
1. **Reported back** (and was honest that it hadn't run that check).

Note, it used the `CLAUDE.md` straight away: the right Python, no pytest, and what the payment types mean.

### The diff

```diff
-def top_tip_zones(df, n=5):
-    df = df.copy()
-    df["tip_pct"] = df["tip_amount"] / df["total_amount"] * 100
+CARD = 1
+
+def top_tip_zones(df, n=5, min_trips=30):
+    df = df[df["payment_type"] == CARD].copy()
+    df["pre_tip"] = df["total_amount"] - df["tip_amount"]
+    df = df[df["pre_tip"] > 0]
+    df["tip_pct"] = df["tip_amount"] / df["pre_tip"] * 100
+    zones = df.groupby("pulocationid")["tip_pct"].agg(["mean", "size"])
+    return (
+        zones.loc[zones["size"] >= min_trips, "mean"]
+             .sort_values(ascending=False)
+             .head(n)
+             .rename("tip_pct")
+    )
```

(docstring left out to fit on the slide)

### Quick quiz

Which of these did the agent decide **without being asked**?

 - A. Only using card trips
 - B. Ignoring zones with fewer than 30 trips
 - C. Changing the tip percentage to be relative to the fare *before* the tip
 - D. Both B and C

**D.** I suggested the cash problem; the other two were its own judgment calls. Both are reasonable, and to its credit it flagged them in its summary ("30 is my judgement call", "goes slightly beyond fix the cash problem"). But they change the answer, and it's your job to notice and decide.

### What did it cost?

Each step sent the whole conversation again. Let's plot the size of the context at each model call:

In [ ]:
calls = {}
for event in session:
    if event['type'] == 'assistant':
        calls[event['message']['id']] = event['message']['usage']  # one entry per model call
usage = pd.DataFrame(calls.values(), index=range(1, len(calls) + 1))
usage.columns = ['new', 'written to cache', 'read from cache']

ax = usage.plot(kind='bar', stacked=True, figsize=(6, 3), rot=0)
ax.set_xlabel('model call')
ax.set_ylabel('input tokens')

In [ ]:
result = session[-1]
total_in = sum(result['usage'][k] for k in ['input_tokens', 'cache_creation_input_tokens', 'cache_read_input_tokens'])
print(f"{result['num_turns']} turns in {result['duration_ms'] / 1000:.0f} s")
print(f"input tokens: {total_in:,}, output tokens: {result['usage']['output_tokens']:,}")
print(f"actual cost: ${result['total_cost_usd']:.2f}")
print(f"without caching: ${total_in * 4 / 1e6 + result['usage']['output_tokens'] * 20 / 1e6:.2f}")

At Opus 5.5's prices ($4 per million input tokens, $20 per million output tokens), caching made this about 3-4 times cheaper. Most of the input is the same system prompt and tools being re-read every step. A long session on a real project can easily be a few dollars; we'll come back to cost in S13.

## Summary

| How it works | What it means for you |
|---|---|
| The model predicts the next token | It produces plausible text, not checked facts |
| It sees tokens, not letters or words | Cost, limits and some odd failures come from tokenization |
| Sampling is random | The same prompt can give different code; don't rely on re-running a conversation |
| Training stops at a knowledge cutoff | Expect outdated or made-up APIs; let it run code and read the docs |
| The whole conversation is resent each turn | Cost grows quickly with session length, and quality drops as the context fills up |
| The model only *asks* for tools | The harness decides what actually runs, so permissions matter |
| The harness shapes the behaviour | Same model, different results; `CLAUDE.md` / `AGENTS.md` are your lever |

# The agentic workflow

So far we've seen *how* an agent works. Now let's look at how to work *with* one.

In Lecture 1 we asked a vague question ("which zones tip the best?") and got a plausible, wrong answer. And in the session we just traced, the agent made two judgment calls of its own that changed the result.

Neither was really the agent's fault. When a request leaves decisions open, the agent has to make them for you, and it will usually do so quietly and reasonably. The way to keep control is to make those decisions yourself, *before* any code gets written.

## Requirements → spec → plan → implement → verify

| Step | The question | Who does the work |
|---|---|---|
| **Requirements** | What does the user actually need, and why? | You, talking to people |
| **Spec** | What exactly should be built: definitions, rules, outputs, what "done" means | You (the agent can help draft and poke holes in it) |
| **Plan** | How will it be built? Which files, which steps, which libraries? | The agent proposes, you review |
| **Implement** | Write the code | The agent |
| **Verify** | Does it meet the spec? | The agent runs the checks, you review the evidence |

Note, this is a loop rather than a straight line: verifying often turns up questions that send you back to the spec.

The middle steps are now cheap and fast, so the steps at either end are where your time goes. A good spec is also exactly what you need to *verify* the result, since it tells you what to check.

## Case study: the same task, two ways

I gave Claude Code (Opus 5.5) the same taxi data and the same `CLAUDE.md` (where Python lives, and that full monthly files are ~7 million rows) in two separate folders, with the same permissions as before.

**Vague:**
> *"Can you write a script that cleans up the taxi data so we can use it for analysis?"*

**Precise:**
> *"Please implement SPEC.md."*

### The spec

```markdown
# Spec: clean_taxis.py

## Goal and user
A cleaned copy of the taxi trips for the team's tip and speed analyses. It will be re-run on each new monthly file.

## Input
`data/taxi_2019.csv` (10,000-row sample). Full monthly files are about 7 million rows with the same columns.

## Output
- `data/taxi_clean.csv`: the kept trips, all original columns, plus `duration_min` and `speed_mph`.
- A printed summary: rows in, rows out, and the number of rows dropped by each rule.
```

```markdown
## Rules
A trip is dropped if any of these apply:
1. Duration under 1 minute or over 3 hours.
2. `trip_distance` is 0 or less, or over 100 miles.
3. `fare_amount` or `total_amount` is 0 or less (refunds and voids).
4. Average speed over 80 mph.
5. `payment_type` is 3 (no charge) or 4 (dispute).

Cash trips (`payment_type` 2) are kept, but their `tip_amount` is set to missing (NaN), because cash tips are not recorded.

## Acceptance criteria
- `test_clean_taxis.py` has a test for each rule and for the cash-tip rule, using small hand-made DataFrames.
- No row in the output breaks any rule.
- Running the script twice gives an identical output file.
- The input file is not modified.

## Performance budget
A 7-million-row month must run in under a minute and 4 GB of memory, so use vectorized pandas only.

## Out of scope
Zone names, and any analysis beyond cleaning.
```

### Quick quiz

What did the **vague** version do with the tips on cash trips?

 - A. Dropped the cash trips altogether
 - B. Set the tips to missing (NaN)
 - C. Left the tips as 0
 - D. Estimated the tips from the card trips

**C.** Lecture 1's bug survives the cleaning. Anyone who averages `tip_amount` over the "clean" data gets $2.64 instead of $3.66 for the trips where a tip was actually recorded.

### What the vague version decided for you

It did a careful job: it explored the data first, wrote tests, and printed what each rule dropped. But it had to make a lot of decisions to do that. From its script and its summary:

 - **Dropped:** exact duplicates; refunds *and* the original charge each one reverses; trips under 0 minutes or over 3 hours; zero distance; over 80 mph; tolls over $100
 - **Changed values:** removed what it decided was a double-counted surcharge from `extra` in about a third of all rows; turned 0 passengers and the "Unknown" zones into missing values
 - **Changed types:** `payment_type` and `ratecodeid` became text labels ("Cash", "JFK", ...), `store_and_fwd_flag` became True/False
 - **Kept:** no-charge and disputed trips, and cash tips as 0

Every one of these is defensible, and it told me about most of them. But they weren't *my* decisions, and some of them break other people's code.

### Quick quiz

Here's the exact line from the vague version's script that relabels the payment types:

In [ ]:
PAYMENT_TYPES = {1: "Credit card", 2: "Cash", 3: "No charge", 4: "Dispute",
                 5: "Unknown", 6: "Voided trip"}

cleaned = taxis.copy()
cleaned["payment_type"] = cleaned["payment_type"].map(PAYMENT_TYPES).astype("category")

A teammate's tip analysis, written against the original data, runs `cleaned[cleaned.payment_type == 1].tip_amount.mean()`. What do they get?

 - A. About 3.71, the same as before
 - B. An error
 - C. `nan`
 - D. 0

In [ ]:
cleaned[cleaned.payment_type == 1].tip_amount.mean()

In [ ]:
(cleaned.payment_type == 1).sum()

No row equals `1` any more, so the selection is empty and the mean of nothing is `nan`. No error and no warning, just like the silent failures we saw in Module A.

### What the spec version did

Let's compare the two sessions using their logs:

In [ ]:
import json

def summarise(path):
    log = [json.loads(line) for line in open(path)]
    result = log[-1]
    tool_calls = sum(block['type'] == 'tool_use'
                     for e in log if e['type'] == 'assistant'
                     for block in e['message']['content'])
    return {
        'turns': result['num_turns'],
        'tool calls': tool_calls,
        'permission denials': sum(e.get('subtype') == 'permission_denied' for e in log),
        'minutes': round(result['duration_ms'] / 60000, 1),
        'cost ($)': round(result['total_cost_usd'], 2),
    }

pd.DataFrame([summarise('../data/claude_code_session_vague.jsonl'),
              summarise('../data/claude_code_session_spec.jsonl')],
             index=['vague', 'spec'])

The spec version took longer and cost a little more, mostly because it actually checked the performance budget: it built a 7-million-row file, found the script used 3.6 GB at its peak, removed two unneeded copies of the table to get it down to 3.2 GB, and timed it at under 30 seconds.

The vague version just said it "should handle the 7M-row files".

It also checked every acceptance criterion, with tests at the edges of each rule (exactly 1 minute, exactly 80 mph...), and then listed what the spec *didn't* cover:

In [ ]:
from IPython.display import Markdown

spec_log = [json.loads(line) for line in open('../data/claude_code_session_spec.jsonl')]
summary = spec_log[-1]['result']
Markdown(summary[summary.index('**Decisions'):])

Those go straight back into the spec as decisions to make. That's the loop from the workflow table.

### The lessons

 - A vague request doesn't get you a worse agent, it gets you the agent's decisions instead of yours.
 - Writing the spec is where your domain knowledge goes in (like knowing that cash tips aren't recorded). The agent can't know that unless the data or you tell it.
 - The acceptance criteria and performance budget gave the agent something to *check*, so it did. Without them, "done" just means "it ran".
 - Both agents still made unrequested decisions; the spec version just made fewer and listed them.

## Writing a spec

There's a template in the course repository at [`project/spec_template.md`](../project/spec_template.md), with the taxi spec as an example. The sections are:

| Section | What goes in it |
|---|---|
| **Goal and user** | Who it's for and what they need it for |
| **Inputs** | The data: source, format, size now and later |
| **Outputs** | Exactly what gets produced |
| **Definitions and rules** | Every decision someone could reasonably make differently |
| **Acceptance criteria** | How you'll know it's done, as things a test can check |
| **Performance budget** | How much data, how fast, how much memory |
| **Out of scope** | What it should *not* do |
| **Open questions** | Decisions you haven't made yet |

The spec is part of Milestone 2. The "done" criteria in this Friday's proposal are a first draft of your acceptance criteria.

### Quick quiz

Which of these is a good acceptance criterion?

 - A. "The output data should be clean"
 - B. "Cash tips are missing (NaN), not 0"
 - C. "The code should be fast"
 - D. "Follow best practices"

**B**, because a test can check it. "Clean", "fast" and "best practices" just hand the decision back to the agent. "Fast" becomes a good criterion once it has numbers: "a 7-million-row month in under a minute".

### Using the agent to help

You don't have to write the spec alone. Some things that work well:

 - Ask the agent to explore the data and **list the decisions** a cleaning script would have to make, *without writing any code*. Then you make them.
 - Ask it to **interview you**: "Ask me questions until you could write a spec for this."
 - Ask for a **plan** before any code. In Claude Code, *plan mode* (Shift+Tab) lets it read files and propose a plan but not change anything until you approve. Other agents have similar modes, or you can just say "don't write any code yet".
 - Ask it to add anything it had to guess to the spec's **open questions**, as the spec version did.

*Tip: keep the spec in the repository (e.g. `SPEC.md`) and point to it from your `CLAUDE.md` or `AGENTS.md`, so every new session starts from the same decisions.*

## Summary

Today, we covered the following topics:
 - Tool calling: the model only asks, and the harness runs the tool
 - The agent loop, and how much a small harness change can fix
 - The anatomy of a harness, and a real Claude Code session step by step
 - The agentic workflow: requirements → spec → plan → implement → verify
 - The same task from a vague request and from a spec
 - Writing a spec, and what makes a good acceptance criterion

Milestone 1 (your proposal) is due Friday at 11:59pm, and there is no class on Friday.

On Monday, we'll look at context management and giving the agent tests it can run, with the same task done with and without a test suite.